# 7.7 Python

Chapter 7 extends the study of joint distributions to continuous settings and introduces several important multivariate families. This section uses Python to work with the Multinomial distribution, the Multivariate Normal, and the Cauchy distribution, building intuition through exact computation and simulation.

In [ ]:
import numpy as np
import scipy.stats as st
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
rng = np.random.default_rng(seed=42)

## Multinomial Distribution

The Multinomial distribution generalizes the Binomial to more than two outcomes. If $n$ independent trials each result in outcome $i$ with probability $p_i$, and $X_i$ counts the number of times outcome $i$ occurs, then $(X_1, \ldots, X_k) \sim \text{Mult}_k(n, \mathbf{p})$. The joint PMF is

$$P(X_1 = x_1, \ldots, X_k = x_k) = \binom{n}{x_1, \ldots, x_k} p_1^{x_1} \cdots p_k^{x_k},$$

provided $x_1 + \cdots + x_k = n$.

`scipy.stats.multinomial` handles this distribution. To evaluate the joint PMF at a specific vector, freeze the distribution with `n` and `p` and then call `.pmf`. As an example, the probability $P(X_1 = 2, X_2 = 0, X_3 = 3)$ for $(X_1, X_2, X_3) \sim \text{Mult}_3(5, (1/3, 1/3, 1/3))$:

In [ ]:
x = [2, 0, 3]
n = 5
p = [1/3, 1/3, 1/3]

dist = st.multinomial(n=n, p=p)
print(f"P(X1=2, X2=0, X3=3) = {dist.pmf(x):.6f}")

# Verify: sum(x) must equal n
print(f"sum(x) = {sum(x)}, n = {n}")

The constraint $\sum x_i = n$ is required by the definition; attempting to evaluate the PMF at a vector whose entries do not sum to $n$ returns zero.

To generate random Multinomial vectors, use `rng.multinomial`. Each call with `size=m` returns an $m \times k$ array in which every row is an independent realization and every row sums to $n$.

In [ ]:
samples = rng.multinomial(n=5, pvals=[1/3, 1/3, 1/3], size=10)

# Display as a table: rows are outcomes, columns are draws (matching R's column convention)
print("Outcome counts per draw (each column is one draw):")
print(samples.T)
print("\nColumn sums (should all be 5):")
print(samples.sum(axis=1))

Each row of `samples` is one draw; transposing arranges the draws as columns, matching the conventional layout. Every row of the transposed display sums to 5, confirming that the counts partition the $n = 5$ trials across the three outcomes.

## Multivariate Normal Distribution

The Multivariate Normal distribution is parameterized by a mean vector $\boldsymbol{\mu}$ and a covariance matrix $\boldsymbol{\Sigma}$. `scipy.stats.multivariate_normal` provides the joint PDF, and `rng.multivariate_normal` generates random vectors.

As a concrete example, consider a Bivariate Normal $(Z, W)$ where both marginals are $N(0, 1)$ and the correlation is $\rho = 0.7$. The covariance matrix is

$$\boldsymbol{\Sigma} = \begin{pmatrix} 1 & \rho \\ \rho & 1 \end{pmatrix},$$

because $\mathrm{Cov}(Z, Z) = \mathrm{Var}(Z) = 1$, $\mathrm{Cov}(W, W) = 1$, and $\mathrm{Cov}(Z, W) = \mathrm{Corr}(Z, W) \cdot \mathrm{SD}(Z) \cdot \mathrm{SD}(W) = \rho$.

In [ ]:
rho = 0.7
mean_vec = [0, 0]
cov = np.array([[1, rho],
                [rho, 1]])

r = rng.multivariate_normal(mean=mean_vec, cov=cov, size=1000)
print(f"Shape of r: {r.shape}  (1000 draws, each a 2-vector)")

fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x=r[:, 0], y=r[:, 1], ax=ax, alpha=0.3, s=12)
ax.set_xlabel("Z")
ax.set_ylabel("W")
ax.set_title(rf"BVN draws ($\rho = {rho}$)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

The cloud of points is visibly elongated along the diagonal, reflecting the strong positive correlation. The aspect ratio is set to 1:1 so the shape is not distorted.

The sample covariance matrix of `r` can be estimated with `np.cov`. Because `np.cov` treats each **row** as a variable, we transpose `r` first so that the two coordinates become the two rows.

In [ ]:
print("Sample covariance matrix:")
print(np.round(np.cov(r.T), 4))
print("\nTrue covariance matrix:")
print(cov)

With 1000 draws the sample covariance matrix is close to the true one; diagonal entries should be near 1 and off-diagonal entries near $\rho = 0.7$.

### Alternative Construction via Linear Combination

Example 7.5.10 describes a way to generate Bivariate Normal draws without using a multivariate generator directly. If $X$ and $Y$ are i.i.d. $N(0, 1)$, then the pair

$$Z = X, \qquad W = \rho X + \tau Y, \quad \tau = \sqrt{1 - \rho^2}$$

follows the same Bivariate Normal distribution. We can verify this by checking that the sample covariance matches.

In [ ]:
tau = np.sqrt(1 - rho**2)

x = rng.normal(size=1000)
y = rng.normal(size=1000)

z = x
w = rho * x + tau * y

r2 = np.column_stack([z, w])   # bind as columns, analogous to cbind(z, w)

print("Sample covariance matrix (linear-combination method):")
print(np.round(np.cov(r2.T), 4))

The result agrees with the direct method. The linear-combination approach makes explicit why the construction works: $\mathrm{Var}(W) = \rho^2 \mathrm{Var}(X) + \tau^2 \mathrm{Var}(Y) = \rho^2 + (1 - \rho^2) = 1$, and $\mathrm{Cov}(Z, W) = \mathrm{Cov}(X,\, \rho X + \tau Y) = \rho \mathrm{Var}(X) = \rho$.

## Cauchy Distribution

The Cauchy distribution (Example 7.1.25) has PDF

$$f(x) = \frac{1}{\pi(1 + x^2)}, \quad -\infty < x < \infty.$$

It is a symmetric, bell-shaped density centered at 0, but its tails are far heavier than those of the Normal. Specifically, neither the mean nor the variance of the Cauchy distribution exists. `scipy.stats.cauchy` provides the PDF and CDF with default location 0 and scale 1.

In [ ]:
print(f"Cauchy PDF at x=0: {st.cauchy.pdf(0):.6f}  (true: 1/π = {1/np.pi:.6f})")
print(f"Cauchy CDF at x=0: {st.cauchy.cdf(0):.6f}  (true: 0.5 by symmetry)")

# Plot the Cauchy PDF
x_grid = np.linspace(-6, 6, 500)
fig, ax = plt.subplots(figsize=(6, 4))
sns.lineplot(x=x_grid, y=st.cauchy.pdf(x_grid), ax=ax)
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title("Cauchy PDF")
plt.tight_layout()
plt.show()

The Cauchy PDF looks superficially like a Normal bell curve, but the polynomial decay $1/(1+x^2)$ is dramatically slower than the Gaussian's exponential decay. This has a striking effect when we try to summarize a sample with a histogram.

With 1000 draws from the Cauchy, a handful of extreme values will appear in the far tails. These outliers force the histogram to span a very wide range, compressing all the central probability mass into a narrow spike and making the histogram look nothing like the smooth PDF above.

In [ ]:
cauchy_draws = rng.standard_cauchy(size=1000)

print(f"Min: {cauchy_draws.min():.1f}")
print(f"Max: {cauchy_draws.max():.1f}")
print(f"Sample mean: {cauchy_draws.mean():.2f}  (undefined in theory)")

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(cauchy_draws, bins=100, ax=ax)
ax.set_xlabel("x")
ax.set_ylabel("Count")
ax.set_title("Histogram of 1000 Cauchy draws — extreme values dominate the x-axis")
plt.tight_layout()
plt.show()

The histogram is almost entirely blank except for a spike near the center, because a few extreme draws spread the x-axis over a very wide range. Clipping the view to the central region reveals the shape more clearly, but also hides the very phenomenon that makes the Cauchy interesting.

In [ ]:
# Clip to the central region to compare with the PDF
x_grid = np.linspace(-10, 10, 500)

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(cauchy_draws, binrange=(-10, 10), bins=60, stat="density", ax=ax, label="Sample histogram")
sns.lineplot(x=x_grid, y=st.cauchy.pdf(x_grid), ax=ax, label="Cauchy PDF")
ax.set_xlim(-10, 10)
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title("Cauchy: histogram (central region) vs. PDF")
ax.legend()
plt.tight_layout()
plt.show()

Even within the clipped window the histogram is noisy compared to, say, 1000 Normal draws. The Cauchy's undefined mean means that the sample mean does not converge to a stable value as the sample size grows — each new draw can shift it dramatically. This is in sharp contrast to distributions with finite means, where the law of large numbers guarantees convergence.